# Decorators Exploration Notebook

This notebook allows you to experiment with decorators interactively.

**Note**: Read the Notes - week 1 - day 1 decorators and create the Jupyter notebook on your own - add logs and experiment on your own

## Basic Decorator

### Manually wrap the function

In [1]:
def my_decorator(func):
    def wrapper():
        print("Something before the function")
        func()
        print("Something after the function")
    
    return wrapper

def say_hello():
    print("Hello")

# manually wrap the function
say_hello = my_decorator(say_hello)
say_hello()

Something before the function
Hello
Something after the function


### With @ Syntax (Preferred)

In [2]:
def my_decorator(func):
    def wrapper():
        print("Something before the function")
        func()
        print("Something after the function")
    
    return wrapper

@my_decorator
def say_hello():
    print("Hello")

say_hello()

Something before the function
Hello
Something after the function


## Decorators with Arguments

In [7]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        print("args are ", args)
        print("kwargs are ", kwargs)
        print(f"Finished {func.__name__}")
        return result
    
    return wrapper

@my_decorator
def add(a,b, word):
    new_word = word + " and " + word
    return a+b, new_word

result = add(5,3, word ="fizz")
print("result = ",result)


Calling add
args are  (5, 3)
kwargs are  {'word': 'fizz'}
Finished add
result =  (8, 'fizz and fizz')


### *args (positional arguments)

args is a tuple containing all positional arguments

In [8]:
def example_args(*args):
    print("args = ",args)
    print("*args = ",*args)

example_args(1, 2)

args =  (1, 2)
*args =  1 2


### *kwargs (keyword arguments)

kwargs is a dictionary mapping argument names to values

In [14]:
def example_kwargs(**kwargs):
    print("kwargs = ",kwargs)
    for key,value in kwargs.items():
        print(f"Key={key} and Value={value}")

example_kwargs(name="John Doe", age=25)

kwargs =  {'name': 'John Doe', 'age': 25}
Key=name and Value=John Doe
Key=age and Value=25


## Using *args and **kwargs together

In [16]:
def example_args_kwargs(*args, **kwargs):
    print("args = ",args)
    print("kwargs = ",kwargs)

example_args_kwargs("a", "b", 1, 2, name="John Doe", age=25 )

args =  ('a', 'b', 1, 2)
kwargs =  {'name': 'John Doe', 'age': 25}


## The functools.wrap Decorator

## without `functools.wraps`

we loose functions metadata (name, docstring, etc)

In [17]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    
    return wrapper

@my_decorator
def greet(name):
    """Greets a person by name"""

    return f"Hello, {name} - wishing you all the very best"

print("greet name ",greet.__name__)
print("greet doc ", greet.__doc__)

greet name  wrapper
greet doc  None


## without `functools.wraps`

we dont loose functions metadata (name, docstring, etc). We preserve function metadata

In [18]:
from functools import wraps

def my_decorator(func):
    @wraps(func)    # This preserves metadata
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    
    return wrapper

@my_decorator
def greet(name):
    """Greets a person by name."""
    return f"Hello, {name}"

print("greet name : ",greet.__name__)
print("greet doc : ",greet.__doc__)

greet name :  greet
greet doc :  Greets a person by name.


>> Always use @wraps(func) in your decorators!

## Common Use cases

### 1. Timing Functions

Its measuring how long a function takes to execute

In [20]:
import time
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"{func.__name__} took {end-start:.4f} seconds")
        return result
    
    return wrapper

@timer
def slow_function():
    time.sleep(1)
    return "Slow function completed"

slow_function()

slow_function took 1.0051 seconds


'Slow function completed'

### 2. Logging

A logging decorator allows us to automatically record information about function calls without adding logging code inside the function itself.

In [21]:
from functools import wraps

def logger(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result}")
    return wrapper

@logger
def add(a,b):
    return a+b

add(5, 3)

Calling add with args=(5, 3), kwargs={}
add returned 8


### 3. Input Validation

In [22]:
from functools import wraps

def validate_positive(func):
    @wraps(func)
    def wrapper(x):
        if x < 0:
            raise ValueError("Number must be positive")
        return func(x)
    return wrapper

@validate_positive
def square_root(x):
    return x**0.5

print(square_root(16))
print(square_root(-4))

4.0


ValueError: Number must be positive

### 4. Caching/Memoization

In [24]:
from functools import wraps

def memoize(func):
    cache = {}

    @wraps(func)
    def wrapper(*args):
        if args in cache:
            print(f"Returning cached result from {args}")
            return cache[args]
        result = func(*args)
        cache[args] = result
        return result
    return wrapper

@memoize
def fibonacci(n):
    if n<2:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

print(fibonacci(10))

Returning cached result from (1,)
Returning cached result from (2,)
Returning cached result from (3,)
Returning cached result from (4,)
Returning cached result from (5,)
Returning cached result from (6,)
Returning cached result from (7,)
Returning cached result from (8,)
55


## Multiple decorators

Order matters! Decorators are applied from bottom to top.

In [26]:
def bold(func):
    def wrapper():
        return "<b>" + func() + "</b>"
    return wrapper

def italic(func):
    def wrapper():
        return "<i>" + func() + "</i>"
    return wrapper

@bold
@italic
def greet():
    return "Hello"

print(greet())


<b><i>Hello</i></b>
